# 10 - Train a Reasoning DQN Model Offline

Same offline loop as `02_train_offline_dqn.ipynb`, with Coconut-style **latent reasoning** added:

1. Every step ends with a **head-output token**: a text const `value` (`format="\n"`, `max_tokens=1`, `head_output: True`). Q-values are read from that token. Reasoning latents are inserted immediately before it.
2. Each training batch samples one **burst step** per sequence (`sample_reasoning_splits`). The model generates `NUM_THOUGHTS` latent "thought" embeddings there, on the autograd tape: each thought's input is the `LatentReasoner` adapter applied to the backbone's output at the previous position.
3. The thoughts are inserted **between the burst step's data tokens and its action prompt**, so the prompt (and every later token in the same task) attends to them.
4. The unchanged `DqnObjective` TD loss backpropagates through the latent chain, training the backbone to emit thoughts that make value predictions more accurate.

Latent generation runs `NUM_THOUGHTS + 1` backbone passes per batch instead of one, so each optimizer step costs roughly `(R + 1)x` the plain forward.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import numpy as np
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount
from mouse_core.models import (
    LatentReasoner,
    Model,
    Polyak,
    push_model_to_hub,
    sample_reasoning_splits,
)
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import RegressionHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-reasoning"    # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-reasoning"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                  # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation
NUM_THOUGHTS = 4                              # latent thoughts generated per reasoning burst


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)


## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.

Same trailing `value` readout as `02` (the head-output / action prompt, here the row-ending newline). The addition here is latent reasoning: at the burst step the thoughts slot in right before that token, `[data tokens][thought 1..R][value]`.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "skip": 0.0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "input_field": "episode_done",
            "format": ",d={field}",
            "skip": 0,
            "format_skipped": "",
        },
        {
            "type": "text",
            "output_field": "value",
            "format": "\n",
            "max_tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
    group_prefix="action,observation,r=reward,d=done\n",
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has a backbone and heads, plus the optional reasoner:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only.
- `RegressionHead` predicts one value per discrete action, read from each step's prompt token.
- `LatentReasoner` is the thought adapter (LayerNorm + Linear): it maps the backbone's output hidden state at the previous position to the input embedding of the next latent thought. `num_thoughts` fixes the burst length `R`.

The backbone exposes `hidden_dim`, and the embedder, head, and reasoner use that same value so the pieces connect cleanly.

`Tokenizer` text fields match `02`: comma-separated action / observation, `r=` / `d=` when nonzero, and a const newline readout flagged `head_output: True`.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. The reasoner is saved and loaded with the checkpoint.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)


head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
)

reasoner = LatentReasoner(hidden_dim=backbone.hidden_dim, num_thoughts=NUM_THOUGHTS)

model = Model(
    backbone=backbone,
    heads={"action_value": head},
    action_source="action_value",
    reasoner=reasoner,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `sample_reasoning_splits(batch=inputs, generator=rng)` picks one burst step per sequence -- uniform over steps whose next step shares the task grouping, so the TD pair out of the burst carries loss weight.
3. `model(inputs, reasoning=splits)` embeds the `TokenBatch`, generates `NUM_THOUGHTS` latent thoughts per burst on the autograd tape (`R` extra backbone passes over the growing prefixes), inserts them before the burst step's first head-output token, and runs the final pass over the extended stream. Predictions keep the flat one-row-per-head-output-token shape, so the objective is unchanged.
4. `objective(objective_data=objective_data, predictions=q, delayed_predictions=q_target)` computes the DQN loss and metrics. Pass `out.predictions["action_value"]` — the name given in `heads=`. TD errors at the burst step and every later same-run step backpropagate through the latent chain.
5. `AdamW` updates weights. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
6. Delayed Q comes from the delayed model: `delayed_model = model.copy(heads=True, backbone=True, reasoner=True)` copies every head, the fp32 backbone (including token embeddings), and the reasoner adapter. `delayed_model(inputs, reasoning=splits)` runs the same `TokenBatch` with the same burst steps, so the delayed model generates its own latent thoughts through its delayed weights. `torch.no_grad()` keeps the target off the tape. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each copied section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`DqnObjective` takes `discount`, called with the unpacked `objective_data` columns (`episode_done` / `task_done` are `0`/`1`/`2`). `boundary_discount` is the standard lookup: `gamma_step` always multiplies, then episode and task extras (`1.0` when the matching code is `0`). When a task ends both extras fire, so a task extra of `0.0` zeros the whole term. `reward=None` and `value=None` skip those callables (raw `reward` column, raw Q). Pass `boundary_reward` / `boundary_value` to transform them. Required `double=False` bootstraps from delayed Q; `double=True` is Double DQN (online Q chooses the action, delayed Q scores it). Required `gate` is the continuation. `gate=None` is the one-step target. Required `bootstrap`: `True` adds `V` where the continuation leaves the sampled run (end of the batch, or a `sequence_id` / `grouping_field` break — a chunk boundary, time limit, or truncation whose rest was not sampled). `False` omits that value. A bootstrap at a state that still has later in-run steps is unchanged, and a true terminal is unchanged either way because its γ already multiplies the value.


In [ ]:
optimizer = AdamW(
    params=model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.copy(heads=True, backbone=True, reasoner=True)
polyak = Polyak(online=model, delayed=delayed_model)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_episode_terminal=1.0,
        gamma_episode_truncated=1.0,
        gamma_task_terminal=0.0,
        gamma_task_truncated=0.0,
    ),
    grouping_field="task_index",
    temperature=0.0,
    double=False,
    bootstrap=True,
    gate=None,
)
rng = np.random.default_rng(0)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        splits = sample_reasoning_splits(batch=inputs, generator=rng)
        out = model(inputs, reasoning=splits)
        with torch.no_grad():
            delayed_out = delayed_model(inputs, reasoning=splits)
        loss, metrics = objective(
            objective_data=to_device(data=objective_data, device=device), predictions=out.predictions["action_value"], delayed_predictions=delayed_out.predictions["action_value"],
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, skips, and `head_output` cannot be recovered from the embedder alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(model=model, tokenizer=tokenizer, repo_id=MODEL_ID, tokenizer_repo_id=TOKENIZER_ID, private=False, clear=True)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")
